In [3]:
import sys  
sys.path.insert(1, '/home/spuchin/GitHub/the-hand-of-midas')

from pandas import DataFrame, read_csv
import plotly.graph_objects as go
from talib._ta_lib import EMA

In [12]:
ohlc: DataFrame = read_csv("/home/spuchin/GitHub/the-hand-of-midas/notebooks/DS-3/macro-micro-candles.csv")
ohlc.head(1)

,open,high,low,close,datetime,open_macro,high_macro,low_macro,close_macro,open_micro,high_micro,low_micro,close_micro
0,4261.48,4349.99,4261.32,4349.99,2017-08-17 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
GLOBAL_EXPONENTIAL_MOVING_AVERAGE: int = 2 ** 10

ohlc["__global_trend"] = (
    EMA(ohlc["high"].values, GLOBAL_EXPONENTIAL_MOVING_AVERAGE) + EMA(ohlc["low"].values, GLOBAL_EXPONENTIAL_MOVING_AVERAGE)
) / 2
ohlc["__global_trend"] = ohlc["__global_trend"].shift(1)

macro_candle_columns = ["open_macro", "high_macro", "low_macro", "close_macro"]
for macro_candle_column in macro_candle_columns:
    ohlc[f"{macro_candle_column}_to_global_trend_ratio"] = (ohlc[macro_candle_column] / ohlc["__global_trend"] - 1)

micro_candle_columns = ["open_micro", "high_micro", "low_micro", "close_micro"]
for micro_candle_column in micro_candle_columns:
    ohlc[f"{micro_candle_column}_to_global_trend_ratio"] = (ohlc[micro_candle_column] / ohlc["__global_trend"] - 1)

noise_candle_columns = ["open", "high", "low", "close"]
for noise_candle_column in noise_candle_columns:
    ohlc["__previous"] = ohlc[noise_candle_column].shift(1)
    ohlc[f"{noise_candle_column}_to_global_trend_ratio"] = (ohlc["__previous"] / ohlc["__global_trend"] - 1)
    ohlc.drop(["__previous"], axis=1, inplace=True)

ohlc.tail(1)

,open,high,low,close,datetime,open_macro,high_macro,low_macro,close_macro,open_micro,...,low_macro_to_global_trend_ratio,close_macro_to_global_trend_ratio,open_micro_to_global_trend_ratio,high_micro_to_global_trend_ratio,low_micro_to_global_trend_ratio,close_micro_to_global_trend_ratio,open_to_global_trend_ratio,high_to_global_trend_ratio,low_to_global_trend_ratio,close_to_global_trend_ratio
17200,107198.02,108135.3,106927.8,107353.08,2025-06-25 12:00:00,103762.253357,104344.666066,103165.923576,103783.979984,104549.577876,...,0.082149,0.088632,0.096662,0.104961,0.092147,0.099929,0.11817,0.124856,0.115475,0.124443


In [14]:
number_of_rows: int = 1200

macro_candlesticks = go.Candlestick(
    x=ohlc["datetime"].tail(number_of_rows),
    open=ohlc["open_macro"].tail(number_of_rows),
    high=ohlc["high_macro"].tail(number_of_rows),
    low=ohlc["low_macro"].tail(number_of_rows),
    close=ohlc["close_macro"].tail(number_of_rows),
    showlegend=False
)
micro_candlesticks = go.Candlestick(
    x=ohlc["datetime"].tail(number_of_rows),
    open=ohlc["open_micro"].tail(number_of_rows),
    high=ohlc["high_micro"].tail(number_of_rows),
    low=ohlc["low_micro"].tail(number_of_rows),
    close=ohlc["close_micro"].tail(number_of_rows),
    increasing_line_color='cyan', decreasing_line_color='blue',
    showlegend=False
)
noise_candlesticks = go.Candlestick(
    x=ohlc["datetime"].tail(number_of_rows),
    open=ohlc["open"].tail(number_of_rows),
    high=ohlc["high"].tail(number_of_rows),
    low=ohlc["low"].tail(number_of_rows),
    close=ohlc["close"].tail(number_of_rows),
    increasing_line_color='gray', decreasing_line_color='black',
    showlegend=False
)
global_trend = go.Scatter(
    x=ohlc["datetime"].tail(number_of_rows),
    y=ohlc["__global_trend"].tail(number_of_rows),
    name="Global Trend Line"
)

figure = go.Figure(data=[noise_candlesticks, micro_candlesticks, macro_candlesticks, global_trend])

figure.update_layout(xaxis_rangeslider_visible=False)
figure.show()

In [15]:
ohlc.drop(["__global_trend"], axis=1, inplace=True)
ohlc.to_csv("global-trend-lines.csv", index=False)

---